<a href="https://colab.research.google.com/github/kousik2006/Titanic-Survival-prediction/blob/main/Titanic_by_pipelining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [46]:
df = pd.read_csv("/content/drive/MyDrive/100 days of ML/Day 29 :- PIpelining/train.csv")

In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [48]:
df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"], inplace= True)

In [49]:
x = df.drop(columns=["Survived"])
y = df['Survived']

In [50]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,y, test_size= 0.2, stratify=y
)

In [51]:
x_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
45,3,male,NaN,0,0,8.050,S
297,1,female,2.0,1,2,151.550,S
768,3,male,NaN,1,0,24.150,Q
736,3,female,48.0,1,3,34.375,S
336,1,male,29.0,1,0,66.600,S


In [52]:
y_train.sample(5)

,Survived
650,0
580,1
636,0
510,1
450,0


In [53]:
from sklearn.compose import ColumnTransformer

In [54]:
from sklearn.impute import SimpleImputer

In [55]:
## imputation transformer

trf1 = ColumnTransformer([
    ("impute_age",SimpleImputer(),[2]),
    ("impute_embarked",SimpleImputer(strategy="most_frequent"),[6])
], remainder= "passthrough")

In [56]:
from sklearn.preprocessing import OneHotEncoder

## 🐛 Bug Fix: `ValueError: could not convert string to float: 'male'`

---

### 📌 The Pipeline Structure

Before diving into the bug, here's a quick recap of what each step in the
pipeline is supposed to do:

| Step | Transformer | Purpose |
|------|-------------|---------|
| `trf1` | `ColumnTransformer` + `SimpleImputer` | Fill missing values in `Age` (mean) and `Embarked` (mode) |
| `trf2` | `ColumnTransformer` + `OneHotEncoder` | Encode categorical columns `Sex` and `Embarked` into numbers |
| `trf3` | `ColumnTransformer` + `MinMaxScaler` | Scale all numeric features to the [0, 1] range |
| `trf4` | `SelectKBest(chi2, k=8)` | Keep only the 8 most statistically significant features |
| `trf5` | `DecisionTreeClassifier` | Train the final model |

Each step feeds its output directly into the next — that's the whole point of
a Pipeline. But this also means **if one step silently produces wrong output,
every step after it breaks too.**

---

### ❌ What Went Wrong

The pipeline crashed at `pipe.fit(x_train, y_train)` with:

```

## 🐛 Bug Fix: `ValueError: could not convert string to float: 'male'`

---

### 📌 The Pipeline Structure

Before diving into the bug, here's a quick recap of what each step in the
pipeline is supposed to do:

| Step | Transformer | Purpose |
|------|-------------|---------|
| `trf1` | `ColumnTransformer` + `SimpleImputer` | Fill missing values in `Age` (mean) and `Embarked` (mode) |
| `trf2` | `ColumnTransformer` + `OneHotEncoder` | Encode categorical columns `Sex` and `Embarked` into numbers |
| `trf3` | `ColumnTransformer` + `MinMaxScaler` | Scale all numeric features to the [0, 1] range |
| `trf4` | `SelectKBest(chi2, k=8)` | Keep only the 8 most statistically significant features |
| `trf5` | `DecisionTreeClassifier` | Train the final model |

Each step feeds its output directly into the next — that's the whole point of
a Pipeline. But this also means **if one step silently produces wrong output,
every step after it breaks too.**

---

### ❌ What Went Wrong

The pipeline crashed at `pipe.fit(x_train, y_train)` with:

```
ValueError: could not convert string to float: 'male'
```

The `Sex` column (containing string values `'male'` and `'female'`) was **never
one-hot encoded**. It passed through `trf2` completely untouched and reached
`SelectKBest(chi2)`, which requires all-numeric input. That's where sklearn
finally threw the error.

But the crash happened at `trf4`, not `trf2`. That's what makes this tricky —
the actual mistake was made two steps earlier, in `trf2`, but there was no
error at that point. It silently passed through the wrong column and only
exploded later.

---

### 🔍 Root Cause: Column Index Drift

This is a **column index drift** problem — a very common silent bug when
chaining multiple `ColumnTransformer` steps.

#### How `ColumnTransformer` reorders columns

When you pass `remainder='passthrough'` to a `ColumnTransformer`, sklearn
outputs columns in this order:

1. The columns you **explicitly specified** (in the order you listed them)
2. The **remaining** columns, in their original left-to-right order

This means the output column order is **different** from the input column order.
Any hardcoded integer indices used in the *next* transformer will now point to
**wrong columns**.

#### Tracing the drift step by step

**Original `x` columns** (after dropping `PassengerId`, `Name`, `Ticket`, `Cabin`):

| Index | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|-------|---|---|---|---|---|---|---|
| Column | Pclass | Sex | Age | SibSp | Parch | Fare | Embarked |

**`trf1`** specifies columns `[2]` (Age) and `[6]` (Embarked) for imputation,
with `remainder='passthrough'` for everything else.

sklearn's output puts the **specified columns first**, then the **rest in order**:

| New Index | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|-----------|---|---|---|---|---|---|---|
| Column | Age | Embarked | Pclass | **Sex** | SibSp | Parch | Fare |

Notice what happened:
- `Age` moved from index 2 → **index 0**
- `Embarked` moved from index 6 → **index 1**
- `Pclass` moved from index 0 → **index 2**
- `Sex` moved from index 1 → **index 3** ⚠️
- `SibSp`, `Parch`, `Fare` all shifted too

**`trf2`** was written using the *original* column indices `[1, 6]`:

```python
trf2 = ColumnTransformer([
    ("ohe_sex_embarked", OneHotEncoder(...), [1, 6])   # intended: Sex, Embarked
], remainder="passthrough")
```

But at this point in the pipeline, index `1` is `Embarked` and index `6` is
`Fare`. So `trf2` actually one-hot encodes **Embarked and Fare** — and `Sex`
(now at index 3) is silently passed through as raw strings.

No error is raised here because sklearn doesn't validate column types inside
`ColumnTransformer` at this step. The bad data just flows forward until
`SelectKBest(chi2)` in `trf4` tries to cast everything to float and finally
crashes.

---

### ✅ The Fix

#### Option 1 — Correct the indices (quick fix)

Update `trf2` to use the **post-trf1 column positions**:

```python
# ❌ Wrong — uses original column indices, before trf1 reorders them
trf2 = ColumnTransformer([
    ("ohe_sex_embarked", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), [1, 6])
], remainder="passthrough")

# ✅ Correct — after trf1, Embarked is at index 1 and Sex is at index 3
trf2 = ColumnTransformer([
    ("ohe_sex_embarked", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), [1, 3])
], remainder="passthrough")
```

This works, but it's fragile — if you ever change `trf1`, you'd have to
manually recalculate all the indices for `trf2` again.

---

#### Option 2 — Use column names with `.set_output()` (recommended)

The proper fix is to stop using integer indices altogether. Add
`.set_output(transform="pandas")` to each `ColumnTransformer` so that the
output is a **DataFrame with column names** instead of a raw numpy array.
This way, the next transformer can reference columns by name and the positions
don't matter at all.

```python
trf1 = ColumnTransformer([
    ("impute_age",      SimpleImputer(),                          ["Age"]),
    ("impute_embarked", SimpleImputer(strategy="most_frequent"),  ["Embarked"])
], remainder="passthrough").set_output(transform="pandas")

trf2 = ColumnTransformer([
    ("ohe_sex_embarked", OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
     ["Sex", "Embarked"])
], remainder="passthrough").set_output(transform="pandas")

trf3 = ColumnTransformer([
    ("scale", MinMaxScaler(), slice(0, 10))
], remainder="passthrough").set_output(transform="pandas")
```

Now `trf2` references `Sex` and `Embarked` by name regardless of how `trf1`
ordered its output — no more silent index drift, no more manual index
calculations when you add or remove a step.

---

### 💡 Key Takeaway

> **Never hardcode integer column indices across chained `ColumnTransformer`
> steps.** `remainder='passthrough'` always reorders output columns, making
> downstream indices invalid. Use column names with `.set_output(transform="pandas")`
> instead — it's one line per transformer and eliminates the entire class of
> index drift bugs.

In [57]:
## one hot encoding

trf2 = ColumnTransformer([
    ("ohe_sex_embarked", OneHotEncoder(sparse_output=False, handle_unknown="ignore"),[1,3])
], remainder="passthrough")

In [58]:
from sklearn.preprocessing import MinMaxScaler

## scaling
# Instead of slice(0,10), we specify the indices of columns to scale.
# We use remainder='passthrough' so the OHE columns are kept but not scaled.
trf3 = ColumnTransformer([
    ("scale", MinMaxScaler(), slice(0,10))
], remainder='passthrough')

In [59]:
from sklearn.feature_selection import SelectKBest, chi2

trf4 = SelectKBest(score_func=chi2, k=8)

In [60]:
## train the model

from sklearn.tree import DecisionTreeClassifier

trf5 = DecisionTreeClassifier()

**Create Pipeline**

In [61]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("trf1", trf1),
    ("trf2", trf2),
    ("trf3", trf3),
    ("trf4", trf4),
    ("trf5", trf5)
])

**Alternative syntex = Make pipeline**

In [62]:
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(trf1,trf2,trf3,trf4,trf5)

In [63]:
## train

pipe.fit(x_train, y_train)

Pipeline(steps=[('columntransformer-1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('columntransformer-2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 3])])),
                ('columntransformer-3',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('selectkbest',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x7a0a8c71cc20>)),
                ('decisiontreeclassifier', DecisionTreeClassifier())])

In [65]:
pipe.named_steps

{'columntransformer-1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'columntransformer-2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 3])]),
 'columntransformer-3': ColumnTransformer(remainder='passthrough',
                   transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'selectkbest': SelectKBest(k=8, score_func=<function chi2 at 0x7a0a8c71cc20>),
 'decisiontreeclassifier': DecisionTreeClassifier()}

In [66]:
## predict

y_pred = pipe.predict(x_test)

In [67]:
y_pred

array([1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1,
       0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0,
       0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0,
       0, 1, 0])

In [68]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.7877094972067039

**Cross validation using pipeline**

In [69]:
from sklearn.model_selection import cross_val_score

cross_val_score(pipe,x_train,y_train,cv=5,scoring="accuracy").mean()

np.float64(0.800640204865557)